# Notebook 1 — EDA: Khám phá tập dữ liệu Pascal VOC 2012

**Bài tập lớn số 2 · CO5085 · HCMUT 2025-2026**

## Mục tiêu
Trong notebook này ta sẽ tìm hiểu:
1. Pascal VOC 2012 là gì? Tại sao chọn dataset này?
2. Phân phối các lớp đối tượng (class distribution)
3. Thống kê bounding boxes: số lượng, kích thước, tỷ lệ khung hình
4. Visualize mẫu ảnh với ground-truth boxes

## Pascal VOC 2012
- 20 lớp đối tượng thường gặp trong ảnh thực tế
- ~11,530 ảnh train / ~2,788 ảnh val
- Annotation dạng XML với bounding boxes (xmin, ymin, xmax, ymax)
- Benchmark chuẩn trong nhiều năm cho object detection

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from torchvision import datasets

from src.data import VOC_CLASSES, get_voc_stats, get_device

print("Device:", get_device())
print("VOC Classes:", len(VOC_CLASSES), "classes")
print(VOC_CLASSES)

## 1. Download và kiểm tra dataset

Torchvision sẽ tự download Pascal VOC 2012 (~2GB) lần đầu tiên chạy.

In [ ]:
# Download dataset (chỉ cần chạy 1 lần)
voc_train = datasets.VOCDetection(root='../data/voc', year='2012',
                                   image_set='train', download=True)
voc_val   = datasets.VOCDetection(root='../data/voc', year='2012',
                                   image_set='val', download=True)

print(f"Train: {len(voc_train)} ảnh")
print(f"Val:   {len(voc_val)} ảnh")

## 2. Thống kê dataset

In [ ]:
stats = get_voc_stats('../data/voc')

print(f"Train images: {stats['n_train']}")
print(f"Val images:   {stats['n_val']}")
print(f"\nBoxes per image: mean={stats.get('boxes_per_image_stats',{}).get('mean','?'):.1f}, "
      f"max={stats.get('boxes_per_image_stats',{}).get('max','?')}")

## 3. Phân phối lớp (Class Distribution)

In [ ]:
class_counts = stats['class_counts']
classes = list(class_counts.keys())
counts = list(class_counts.values())

# Sắp xếp theo số lượng giảm dần
sorted_pairs = sorted(zip(counts, classes), reverse=True)
counts_sorted, classes_sorted = zip(*sorted_pairs)

fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(classes)))
bars = ax.bar(classes_sorted, counts_sorted, color=colors)
ax.set_xticklabels(classes_sorted, rotation=45, ha='right')
ax.set_ylabel('Số instances')
ax.set_title('Phân phối các lớp trong Pascal VOC 2012 (Train+Val)')
ax.grid(axis='y', alpha=0.3)

# Thêm số trên mỗi bar
for bar, cnt in zip(bars, counts_sorted):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(cnt), ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('../results/plots/class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nNhận xét: 'person' chiếm đa số (~30%), một số lớp hiếm như 'boat', 'sheep'")

## 4. Thống kê Bounding Boxes

In [ ]:
from xml.etree import ElementTree as ET
from pathlib import Path

voc_base = Path('../data/voc/VOCdevkit/VOC2012')
ann_dir = voc_base / 'Annotations'

all_widths, all_heights, boxes_per_img = [], [], []

for ann_file in list(ann_dir.glob('*.xml'))[:3000]:  # Sample 3000 ảnh cho nhanh
    tree = ET.parse(ann_file)
    root = tree.getroot()
    size = root.find('size')
    img_w = float(size.find('width').text)
    img_h = float(size.find('height').text)

    objs = root.findall('object')
    boxes_per_img.append(len(objs))
    for obj in objs:
        bb = obj.find('bndbox')
        w = float(bb.find('xmax').text) - float(bb.find('xmin').text)
        h = float(bb.find('ymax').text) - float(bb.find('ymin').text)
        all_widths.append(w / img_w)   # normalized
        all_heights.append(h / img_h)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(boxes_per_img, bins=20, color='#6366f1', edgecolor='white')
axes[0].set_title('Số boxes mỗi ảnh')
axes[0].set_xlabel('Số bounding boxes')
axes[0].set_ylabel('Số ảnh')

axes[1].hist(all_widths, bins=50, color='#ec4899', edgecolor='white')
axes[1].set_title('Chiều rộng boxes (normalized)')
axes[1].set_xlabel('Width / Image Width')

axes[2].scatter(all_widths[:1000], all_heights[:1000], alpha=0.3, s=10, color='#10b981')
axes[2].set_title('Width vs Height (normalized)')
axes[2].set_xlabel('Width')
axes[2].set_ylabel('Height')

plt.tight_layout()
plt.savefig('../results/plots/bbox_stats.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Visualize mẫu ảnh với Ground-Truth Boxes

In [ ]:
from src.utils import visualize_detections
from src.data import VOCDetectionDataset, get_val_transforms

ds = VOCDetectionDataset('../data/voc', year='2012', image_set='val',
                          transforms=get_val_transforms())

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, ax in enumerate(axes):
    img_t, target = ds[i * 30]
    visualize_detections(
        image=img_t,
        boxes=target['boxes'].numpy().tolist(),
        labels=target['labels'].numpy().tolist(),
        gt_boxes=None,
        title=f"Val image #{i*30}",
        ax=ax,
    )

plt.suptitle('Mẫu ảnh Pascal VOC 2012 với Ground-Truth Boxes', fontsize=12)
plt.tight_layout()
plt.savefig('../results/plots/sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

## Tóm tắt

| Đặc điểm | Giá trị |
|-----------|---------|
| Số lớp | 20 |
| Train images | ~11,530 |
| Val images | ~2,788 |
| Boxes/image (trung bình) | ~2.7 |
| Lớp phổ biến nhất | person |
| Lớp ít nhất | (xem biểu đồ) |

**Lý do chọn Pascal VOC 2012:**
- Kích thước vừa phải, phù hợp để fine-tune trong thời gian hợp lý
- 20 lớp đa dạng, đủ khó để đánh giá model
- Benchmark lâu đời, kết quả so sánh được với nhiều paper
- Torchvision hỗ trợ sẵn (không cần tự parse)